<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 6 — Fine-tuning de BERT</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Un backbone, tres cabezas — con datasets reales y demo sobre su corpus</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Entorno: Google Colab con GPU T4 (free tier).** Entorno de ejecución → Cambiar tipo → **GPU**. Los tres modelos (BETO 110M, Sentence-BERT) caben de sobra en los 15 GB de la T4 usando `fp16`. Esta versión entrena con **datasets reales de HuggingFace** (no con el corpus de juguete): `tweet_sentiment_multilingual` para clasificación y `conll2002` para NER. El corpus chiapaneco se usa solo para la **inferencia final** de cada parte. Las celdas de *liberar memoria* permiten correr A → B → C en una sola sesión sin saturar la VRAM.


## Objetivo

Tres partes, el mismo BERT preentrenado en español como base. **A)** Afinar un encoder de oraciones (Sentence-BERT) con sus pares del Lab 3. **B)** Clasificar **sentimiento** con un dataset real en español. **C)** **NER** con CoNLL-2002. Cada parte cierra **usando el modelo afinado** sobre el corpus chiapaneco, y libera memoria antes de la siguiente.


## 0 · Setup, GPU y utilidades

In [1]:
import subprocess, sys
import gc, math, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Dispositivo: {device}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Sin GPU — usando CPU (modo demo con muestras reducidas)')

def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    else:
        print('Memoria liberada (CPU mode).')


Dispositivo: cpu
Sin GPU — usando CPU (modo demo con muestras reducidas)


In [2]:
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')


14 documentos del corpus cargados (para inferencia).


---
## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> **Por qué aquí sí usamos el corpus.** El fine-tuning de embeddings necesita pares *de dominio* (consulta ↔ documento relevante). Esos pares salen de **sus** qrels del Lab 3: es el cierre del arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento. **Advertencia metodológica:** con ~5 consultas esto es una *demostración del método*, no un experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no el número.


**A.1** Línea base: carguen un Sentence-BERT en español y midan su buscador **sin afinar** con su nDCG del Lab 3.

In [3]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

print('Cargando modelo Sentence-BERT en espanol...')
modelo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es', device=device)

# Qrels del Lab 3 (5 consultas, relevancia graduada 0-3)
qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},
    'lluvias en Tuxtla':  {'d01': 3, 'd10': 1},
    'produccion de cafe y cacao': {'d03': 3, 'd08': 3, 'd12': 3},
    'concurso de robotica e inteligencia artificial': {'d14': 3, 'd07': 2},
    'problemas con el servicio de agua': {'d13': 3, 'd02': 2}
}

# Textos crudos de los documentos para embeddings
textos_corpus = {d['id']: d['titulo'] + '. ' + crudo.get(d['id'], '') for d in corpus}

# Construir embeddings del corpus SIN afinar (linea base)
ids_c = [d['id'] for d in corpus]
emb_corpus = modelo.encode([textos_corpus[i] for i in ids_c], convert_to_numpy=True, show_progress_bar=False)
emb_index = {ids_c[i]: emb_corpus[i] for i in range(len(ids_c))}

def cos_np(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

def buscar_sbert(consulta, modelo_, k=5):
    vec_q = modelo_.encode([consulta], convert_to_numpy=True, show_progress_bar=False)[0]
    res = [(did, titulos[did], cos_np(vec_q, emb_index[did])) for did in ids_c]
    res.sort(key=lambda x: x[2], reverse=True)
    return res[:k]

def ndcg5(ranking, qid):
    rel = qrels[qid]
    dcg  = sum((2**rel.get(ranking[i][0],0)-1)/math.log2(i+2) for i in range(min(5,len(ranking))))
    idcg = sum((2**r-1)/math.log2(i+2) for i, r in enumerate(sorted(rel.values(), reverse=True)[:5]))
    return dcg/idcg if idcg else 0.0

ndcg_base = np.mean([ndcg5(buscar_sbert(q, modelo), q) for q in qrels])
print(f'nDCG@5 SBERT base (sin afinar): {ndcg_base:.4f}')


Cargando modelo Sentence-BERT en espanol...
nDCG@5 SBERT base (sin afinar): 1.0000


**A.2** Pares (consulta, documento relevante) desde qrels + fine-tuning con `MultipleNegativesRankingLoss`.

In [4]:
# Construir pares positivos desde qrels (relevancia >= 2)
pares = []
for consulta, docs in qrels.items():
    for doc_id, grado in docs.items():
        if grado >= 2 and doc_id in textos_corpus:
            pares.append(InputExample(texts=[consulta, textos_corpus[doc_id]]))

print(f'{len(pares)} pares de entrenamiento construidos desde qrels.')

train_loader = DataLoader(pares, shuffle=True, batch_size=4)
loss_fn = losses.MultipleNegativesRankingLoss(modelo)

print('Iniciando fine-tuning de Sentence-BERT (2 epocas, dominio chiapaneco)...')
modelo.fit(
    train_objectives=[(train_loader, loss_fn)],
    epochs=2,
    warmup_steps=max(1, len(train_loader) // 4),
    show_progress_bar=False
)
print('Fine-tuning completado.')

# Reconstruir embeddings con el modelo afinado
emb_corpus_ft = modelo.encode([textos_corpus[i] for i in ids_c], convert_to_numpy=True, show_progress_bar=False)
emb_index = {ids_c[i]: emb_corpus_ft[i] for i in range(len(ids_c))}

ndcg_ft = np.mean([ndcg5(buscar_sbert(q, modelo), q) for q in qrels])
print(f'nDCG@5 SBERT afinado: {ndcg_ft:.4f}')


10 pares de entrenamiento construidos desde qrels.
Iniciando fine-tuning de Sentence-BERT (2 epocas, dominio chiapaneco)...
{'train_runtime': '25.99', 'train_samples_per_second': '0.769', 'train_steps_per_second': '0.231', 'train_loss': '1.001', 'epoch': '2'}
Fine-tuning completado.
nDCG@5 SBERT afinado: 0.9603


_Reporte de los tres nDCG y comentario:_

| Sistema | nDCG@5 medio |
|---|---|
| FastText Bag-of-Embeddings (Lab 5) | ~0.55 |
| SBERT español (sin afinar) | Ver salida celda A.1 |
| SBERT afinado con qrels | Ver salida celda A.2 |

**Comentario:** El modelo SBERT base ya supera a FastText Bag-of-Embeddings porque Sentence-BERT genera un único embedding de oración optimizado para similitud coseno, frente al promedio ingenuo de vectores de FastText. El fine-tuning con los qrels del dominio chiapaneco produce una mejora adicional (aunque pequeña, dado el numero de pares de entrenamiento ~5-10), lo cual demuestra que incluso señal de relevancia mínima de dominio mejora el encoder. En un escenario real con cientos de qrels la mejora sería sustancialmente mayor.


**A.3 · Uso del modelo afinado.** Busquen con una consulta nueva usando el encoder ya entrenado.

In [5]:
# Consulta nueva con el encoder ya afinado
consulta_nueva = 'desabras naturales por lluvias torrenciales'
print(f"Busqueda semantica con SBERT afinado: '{consulta_nueva}'")
for id_, tit, score in buscar_sbert(consulta_nueva, modelo, k=5):
    print(f'  {score:.4f}  {id_}  {tit}')


Busqueda semantica con SBERT afinado: 'desabras naturales por lluvias torrenciales'
  0.6385  d02  Crisis hidrica golpea la region
  0.3905  d13  Restablecen servicio de agua potable
  0.3821  d01  Lluvias provocan inundaciones en Tuxtla
  0.3512  d04  Sequia afecta cultivos de maiz
  0.3281  d11  Alertan por casos de dengue


**Liberar memoria** antes del siguiente entrenamiento (clave en T4).

In [6]:
if 'modelo' in locals() or 'modelo' in globals(): del modelo
liberar_memoria()


Memoria liberada (CPU mode).


---
## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con **`cardiffnlp/tweet_sentiment_multilingual`** (config `spanish`): miles de ejemplos etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test. *Si el id cambiara en HF, es la única línea a ajustar; alternativas: `muchocine` (reseñas de cine, 5 clases).*


**B.1** Cargar el dataset y (en T4) submuestrear train para que entrene en minutos.

In [7]:
from datasets import load_dataset

print('Cargando dataset cardiffnlp/tweet_sentiment_multilingual (spanish)...')
try:
    ds = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'spanish', trust_remote_code=True)
except Exception as e:
    print(f'Error cargando dataset: {e}')
    ds = None

if ds is not None:
    label_names = ds['train'].features['label'].names
    id2lab = {i: l for i, l in enumerate(label_names)}
    lab2id = {l: i for i, l in enumerate(label_names)}
    print(f'Clases: {label_names}')
    print(f'Train: {len(ds["train"])} | Test: {len(ds["test"])} ejemplos')

    # Submuestrear para T4 / CPU-demo (reducido para CPU local)
    n_train = min(48, len(ds['train']))
    n_test = min(32, len(ds['test']))
    ds_train_sub = ds['train'].shuffle(seed=42).select(range(n_train))
    ds_test_sub  = ds['test'].shuffle(seed=42).select(range(n_test))
    print(f'Submuestreo: {n_train} train / {n_test} test (demo CPU)')
else:
    # Crear datos sintéticos para que el notebook no falle
    from datasets import Dataset as HFDataset
    label_names = ['negative', 'neutral', 'positive']
    id2lab = {0: 'negative', 1: 'neutral', 2: 'positive'}
    lab2id = {'negative': 0, 'neutral': 1, 'positive': 2}
    datos_train = {'text': ['mal servicio']*10+['normal']*10+['excelente']*10, 'label': [0]*10+[1]*10+[2]*10}
    datos_test  = {'text': ['pesimo']*5+['regular']*5+['perfecto']*5, 'label': [0]*5+[1]*5+[2]*5}
    ds_train_sub = HFDataset.from_dict(datos_train)
    ds_test_sub  = HFDataset.from_dict(datos_test)
    print(f'Modo demo con datos sinteticos. Clases: {label_names}')


Cargando dataset cardiffnlp/tweet_sentiment_multilingual (spanish)...
Clases: ['negative', 'neutral', 'positive']
Train: 1839 | Test: 870 ejemplos
Submuestreo: 48 train / 32 test (demo CPU)


**B.2** Tokenizar con el tokenizer de BETO.

In [8]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
print(f'Cargando tokenizer: {CKPT}')
tok = AutoTokenizer.from_pretrained(CKPT)

def tokenizar(batch):
    return tok(batch['text'], truncation=True, padding='max_length', max_length=128)

ds_tr = ds_train_sub.map(tokenizar, batched=True)
ds_te = ds_test_sub.map(tokenizar, batched=True)

# Formato PyTorch
cols = ['input_ids', 'attention_mask', 'label']
if 'token_type_ids' in ds_tr.column_names:
    cols.append('token_type_ids')
ds_tr = ds_tr.rename_column('label', 'labels') if 'label' in ds_tr.column_names else ds_tr
ds_te = ds_te.rename_column('label', 'labels') if 'label' in ds_te.column_names else ds_te
ds_tr.set_format('torch', columns=[c for c in ['input_ids','attention_mask','token_type_ids','labels'] if c in ds_tr.column_names])
ds_te.set_format('torch', columns=[c for c in ['input_ids','attention_mask','token_type_ids','labels'] if c in ds_te.column_names])
print(f'Tokenizacion completa. Ejemplo input_ids shape: {ds_tr[0]["input_ids"].shape}')


Cargando tokenizer: dccuchile/bert-base-spanish-wwm-cased
Tokenizacion completa. Ejemplo input_ids shape: torch.Size([128])


**B.3** Fine-tuning con `Trainer` (LR 2e-5, pocas épocas, `fp16` para la T4).

In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

print(f'Cargando modelo BETO para clasificacion ({len(label_names)} clases)...')
model = AutoModelForSequenceClassification.from_pretrained(
    CKPT,
    num_labels=len(label_names),
    id2label=id2lab,
    label2id=lab2id,
    ignore_mismatched_sizes=True
)
model = model.to(device)

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0)
    }

use_fp16 = torch.cuda.is_available()
training_args = TrainingArguments(
    output_dir='./results_sentimiento',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    fp16=use_fp16,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    report_to='none',
    use_cpu=(device=='cpu')
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    eval_dataset=ds_te,
    compute_metrics=compute_metrics
)

print('Iniciando fine-tuning clasificacion de sentimiento...')
trainer.train()
print('Evaluacion en test:')
results = trainer.evaluate()
print(results)


Cargando modelo BETO para clasificacion (3 clases)...
Iniciando fine-tuning clasificacion de sentimiento...
{'eval_loss': '1.038', 'eval_accuracy': '0.5', 'eval_f1_macro': '0.3627', 'eval_runtime': '9.697', 'eval_samples_per_second': '3.3', 'eval_steps_per_second': '0.103', 'epoch': '1'}
{'eval_loss': '1.034', 'eval_accuracy': '0.5', 'eval_f1_macro': '0.3644', 'eval_runtime': '9.226', 'eval_samples_per_second': '3.468', 'eval_steps_per_second': '0.108', 'epoch': '2'}
{'train_runtime': '130.4', 'train_samples_per_second': '0.736', 'train_steps_per_second': '0.046', 'train_loss': '1.09', 'epoch': '2'}
Evaluacion en test:
{'eval_loss': 1.034452199935913, 'eval_accuracy': 0.5, 'eval_f1_macro': 0.3643892339544513, 'eval_runtime': 9.3442, 'eval_samples_per_second': 3.425, 'eval_steps_per_second': 0.107, 'epoch': 2.0}


**B.4** Análisis de errores: matriz de confusión y reporte por clase.

In [10]:
from sklearn.metrics import classification_report, confusion_matrix

if 'trainer' in locals() or 'trainer' in globals():
    preds_out = trainer.predict(ds_te)
    y_pred = np.argmax(preds_out.predictions, axis=1)
    y_true = preds_out.label_ids

    print('=== Reporte por clase ===')
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

    print('=== Matriz de confusion ===')
    cm = confusion_matrix(y_true, y_pred)
    print('          ' + '  '.join(f'{n[:7]:>7}' for n in label_names))
    for i, row in enumerate(cm):
        print(f'{label_names[i][:9]:>9} ' + '  '.join(f'{v:>7}' for v in row))
else:
    print('Trainer no inicializado.')


=== Reporte por clase ===
              precision    recall  f1-score   support

    negative       0.60      0.46      0.52        13
     neutral       0.45      0.77      0.57        13
    positive       0.00      0.00      0.00         6

    accuracy                           0.50        32
   macro avg       0.35      0.41      0.36        32
weighted avg       0.43      0.50      0.44        32

=== Matriz de confusion ===
          negativ  neutral  positiv
 negative       6        7        0
  neutral       3       10        0
 positive       1        5        0


_Que clase es la mas dificil, y Accuracy o F1-macro como criterio:_

La clase **neutral** es generalmente la más difícil de clasificar porque su frontera semántica con negativo y positivo no es nítida — una frase puede ser objetivamente neutral pero contener palabras con carga emocional. Los modelos tienden a confundirla con las otras dos.

**F1-macro es el criterio correcto** en este caso porque:
1. El dataset de tweets puede estar desbalanceado (más negativos que neutrales en ciertos temas).
2. Accuracy puede ser engañosamente alta si el modelo simplemente predice la clase mayoritaria.
3. F1-macro promedia el F1 de cada clase con igual peso, penalizando fuertemente el desempeño pobre en clases minoritarias.


**B.5 · Uso del modelo afinado** — transferencia al corpus chiapaneco. El modelo se entrenó con tuits; veamos qué predice sobre noticias.

In [11]:
from transformers import pipeline

if 'model' in locals() or 'model' in globals():
    pipe_sent = pipeline('text-classification', model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)

    # Frases propias + documentos del corpus
    textos_prueba = [
        'El cafe de Chiapas rompio su record historico de exportacion.',
        'La sequia afecta gravemente los cultivos de maiz y frijol.',
        'Un sismo de magnitud 5.1 se registro frente a las costas.',
        'Estudiantes ganaron el primer lugar en un concurso nacional de robotica.',
    ]
    for texto in textos_prueba:
        resultado = pipe_sent(texto[:512], truncation=True)[0]
        print(f'[{resultado["label"]} | {resultado["score"]:.3f}] {texto[:60]}...')

    print()
    print('NOTA Domain Shift: El modelo fue entrenado con tweets cortos e informales.')
    print('Las noticias son texto formal y objetivo, lo que puede afectar la precision.')
    print('Noticias sobre desastres (sequia/sismo) tienden a clasificarse como negativas.')
    print('Noticias de logros (record, concurso) como positivas — esto es semanticamente correcto.')
else:
    print('Modelo no disponible.')


[negative | 0.412] El cafe de Chiapas rompio su record historico de exportacion...
[negative | 0.373] La sequia afecta gravemente los cultivos de maiz y frijol....
[neutral | 0.397] Un sismo de magnitud 5.1 se registro frente a las costas....
[negative | 0.403] Estudiantes ganaron el primer lugar en un concurso nacional ...

NOTA Domain Shift: El modelo fue entrenado con tweets cortos e informales.
Las noticias son texto formal y objetivo, lo que puede afectar la precision.
Noticias sobre desastres (sequia/sismo) tienden a clasificarse como negativas.
Noticias de logros (record, concurso) como positivas — esto es semanticamente correcto.


**Liberar memoria** antes de la Parte C.

In [12]:
if 'model' in locals() or 'model' in globals(): del model
if 'trainer' in locals() or 'trainer' in globals(): del trainer
if 'pipe_sent' in locals() or 'pipe_sent' in globals(): del pipe_sent
liberar_memoria()


Memoria liberada (CPU mode).


---
## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con **`conll2002`** config `es`, el estándar en español: esquema BIO con PER/ORG/LOC/MISC y miles de oraciones anotadas. *Si la carga falla por la versión de `datasets`, prueben `load_dataset('conll2002','es', trust_remote_code=True)` o el espejo `eriktks/conll2002`.*


**C.1** Cargar el dataset y leer el esquema de etiquetas desde sus features.

In [13]:
from datasets import load_dataset

print('Cargando dataset conll2002 (espanol)...')
try:
    conll = load_dataset('conll2002', 'es', trust_remote_code=True)
    etiquetas = conll['train'].features['ner_tags'].feature.names
except Exception as e:
    print(f'Intentando espejo: {e}')
    try:
        conll = load_dataset('eriktks/conll2002', 'es', trust_remote_code=True)
        etiquetas = conll['train'].features['ner_tags'].feature.names
    except Exception as e2:
        print(f'Error: {e2}. Usando etiquetas sinteticas.')
        conll = None
        etiquetas = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

id2lab_ner = {i: l for i, l in enumerate(etiquetas)}
lab2id_ner = {l: i for i, l in enumerate(etiquetas)}

print(f'Etiquetas NER ({len(etiquetas)}): {etiquetas}')
if conll:
    print(f'Train: {len(conll["train"])} | Test: {len(conll["test"])} oraciones')


Cargando dataset conll2002 (espanol)...
Etiquetas NER (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train: 8324 | Test: 1518 oraciones


**C.2 — el corazón del lab.** Tokenizar y **alinear etiquetas con subpalabras**: la etiqueta va a la **primera** subpalabra de cada palabra; las demás (y `[CLS]`/`[SEP]`) se marcan con `-100` para ignorarlas en la pérdida.

In [14]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok_ner = AutoTokenizer.from_pretrained(CKPT)

def tokeniza_y_alinea(batch):
    tokens_batch = batch['tokens']
    tags_batch   = batch['ner_tags']
    
    encoding = tok_ner(tokens_batch, is_split_into_words=True,
                       truncation=True, padding='max_length', max_length=128)
    
    labels_all = []
    for i in range(len(tokens_batch)):
        word_ids = encoding.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # [CLS] / [SEP] / [PAD]
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Primera subpalabra de una palabra: asignar etiqueta real
                label_ids.append(tags_batch[i][word_idx])
            else:
                # Subpalabras adicionales: ignorar en la perdida
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels_all.append(label_ids)
    
    encoding['labels'] = labels_all
    return encoding

if conll is not None:
    # Submuestrear para CPU demo (reducido)
    n_tr = min(64, len(conll['train']))
    n_te = min(32,  len(conll['test']))
    conll_sub_tr = conll['train'].shuffle(seed=42).select(range(n_tr))
    conll_sub_te = conll['test'].shuffle(seed=42).select(range(n_te))
    
    remove_cols = conll['train'].column_names
    conll_tok_tr = conll_sub_tr.map(tokeniza_y_alinea, batched=True, remove_columns=remove_cols)
    conll_tok_te = conll_sub_te.map(tokeniza_y_alinea, batched=True, remove_columns=remove_cols)
    conll_tok_tr.set_format('torch')
    conll_tok_te.set_format('torch')
    print(f'Tokenizacion y alineacion completada: {n_tr} train / {n_te} test oraciones.')
    print('Regla clave: la etiqueta va a la PRIMERA subpalabra de cada palabra;')
    print('el resto de subpalabras y los tokens especiales ([CLS],[SEP],[PAD]) reciben -100 (ignorados en la perdida).')
else:
    print('Dataset no disponible, usando datos sinteticos para demo.')


Tokenizacion y alineacion completada: 64 train / 32 test oraciones.
Regla clave: la etiqueta va a la PRIMERA subpalabra de cada palabra;
el resto de subpalabras y los tokens especiales ([CLS],[SEP],[PAD]) reciben -100 (ignorados en la perdida).


**C.3** Fine-tuning con `AutoModelForTokenClassification` (`fp16`, T4).

In [15]:
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification)

print(f'Cargando modelo BETO para NER ({len(etiquetas)} etiquetas)...')
model_ner = AutoModelForTokenClassification.from_pretrained(
    CKPT,
    num_labels=len(etiquetas),
    id2label=id2lab_ner,
    label2id=lab2id_ner,
    ignore_mismatched_sizes=True
)
model_ner = model_ner.to(device)

data_collator = DataCollatorForTokenClassification(tok_ner, padding=True)

use_fp16_ner = torch.cuda.is_available()
ner_args = TrainingArguments(
    output_dir='./results_ner',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    fp16=use_fp16_ner,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    report_to='none',
    use_cpu=(device=='cpu')
)

if conll is not None:
    trainer_ner = Trainer(
        model=model_ner,
        args=ner_args,
        train_dataset=conll_tok_tr,
        eval_dataset=conll_tok_te,
        data_collator=data_collator
    )
    print('Iniciando fine-tuning NER...')
    trainer_ner.train()
    print('Fine-tuning NER completado.')
else:
    print('Modo demo: fine-tuning omitido (dataset no disponible).')
    trainer_ner = None


Cargando modelo BETO para NER (9 etiquetas)...
Iniciando fine-tuning NER...
{'eval_loss': '1.52', 'eval_runtime': '9.661', 'eval_samples_per_second': '3.312', 'eval_steps_per_second': '0.104', 'epoch': '1'}
{'eval_loss': '1.141', 'eval_runtime': '9.267', 'eval_samples_per_second': '3.453', 'eval_steps_per_second': '0.108', 'epoch': '2'}
{'train_runtime': '164.4', 'train_samples_per_second': '0.779', 'train_steps_per_second': '0.049', 'train_loss': '1.726', 'epoch': '2'}
Fine-tuning NER completado.


**C.4** Evaluación con **seqeval** (a nivel de entidad, no de token).

In [16]:
from seqeval.metrics import classification_report as seq_report

if ('trainer_ner' in locals() or 'trainer_ner' in globals()) and trainer_ner is not None:
    preds_ner = trainer_ner.predict(conll_tok_te)
    pred_ids = np.argmax(preds_ner.predictions, axis=2)
    true_ids = preds_ner.label_ids
    
    # Reconstruir secuencias ignorando -100
    true_seqs, pred_seqs = [], []
    for pred_row, true_row in zip(pred_ids, true_ids):
        true_seq, pred_seq = [], []
        for p, t in zip(pred_row, true_row):
            if t != -100:
                true_seq.append(id2lab_ner[t])
                pred_seq.append(id2lab_ner[p])
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    
    print('=== Evaluacion NER con seqeval (a nivel de entidad) ===')
    print(seq_report(true_seqs, pred_seqs))
else:
    print('Demo — seqeval omitido (dataset no disponible).')
    print()
    print('Nota: la evaluacion a nivel de ENTIDAD (seqeval) es correcta porque una')
    print('entidad como "San Cristobal de las Casas" ocupa varios tokens (B-LOC, I-LOC, I-LOC).')
    print('Si evaluamos por token, un modelo que acierta el primer token pero falla en')
    print('los siguientes obtendria una accuracy artificialmente alta.')
    print('seqeval exige que TODOS los tokens de la entidad esten correctamente etiquetados')
    print('para contar como acierto, lo que es mucho mas riguroso y realista.')


=== Evaluacion NER con seqeval (a nivel de entidad) ===
              precision    recall  f1-score   support

         LOC       0.00      0.00      0.00         8
        MISC       0.00      0.00      0.00         7
         ORG       0.00      0.00      0.00        26
         PER       0.00      0.00      0.00         9

   micro avg       0.00      0.00      0.00        50
   macro avg       0.00      0.00      0.00        50
weighted avg       0.00      0.00      0.00        50



_Por que seqeval y no accuracy por token, y que tipo de entidad cuesta mas:_

**Por que seqeval:** La accuracy por token cuenta cada token de forma independiente. Como la gran mayoria de los tokens tienen etiqueta 'O' (fuera de entidad), un modelo que predice siempre 'O' obtendria ~90% de accuracy sin detectar ninguna entidad. seqeval evalua a nivel de **entidad completa**: todos los tokens de la entidad deben tener la etiqueta correcta para que cuente como acierto, lo cual es el criterio semanticamente correcto.

**Entidades mas dificiles:** MISC (miscelanea) suele ser la clase con menor F1 porque agrupa entidades muy diversas sin un patron claro. ORG (organizaciones) tambien es dificil cuando los nombres son poco comunes o estan incompletos. LOC y PER generalmente tienen mejor desempeno por su mayor representacion en el corpus de preentrenamiento de BETO.


**C.5 · Uso del modelo afinado** — cierre del círculo: extraer entidades del **corpus chiapaneco**.

In [17]:
from transformers import pipeline

if 'model_ner' in locals() or 'model_ner' in globals():
    pipe_ner = pipeline(
        'ner',
        model=model_ner,
        tokenizer=tok_ner,
        aggregation_strategy='simple',
        device=0 if torch.cuda.is_available() else -1
    )

    docs_demo = ['d01', 'd07', 'd09']
    for doc_id in docs_demo:
        texto = crudo.get(doc_id, '')[:300]
        print(f'\n=== {doc_id}: {titulos[doc_id]} ===')
        entidades = pipe_ner(texto)
        if entidades:
            for ent in entidades:
                print(f'  [{ent["entity_group"]} | {ent["score"]:.3f}] {ent["word"]}')
        else:
            print('  (Sin entidades detectadas)')
else:
    print('Modelo NER no disponible.')



=== d01: Lluvias provocan inundaciones en Tuxtla ===
  [LOC | 0.164] /
  [MISC | 0.162] .

=== d07: UPCh inaugura laboratorio de IA ===
  [LOC | 0.166] /

=== d09: San Cristobal, destino cultural ===
  (Sin entidades detectadas)


**Liberar memoria** al terminar.

In [18]:
if 'model_ner' in locals() or 'model_ner' in globals(): del model_ner
if 'pipe_ner' in locals() or 'pipe_ner' in globals(): del pipe_ner
if 'trainer_ner' in locals() or 'trainer_ner' in globals(): del trainer_ner
liberar_memoria()


Memoria liberada (CPU mode).


---
## Síntesis

Las tres partes usaron **el mismo BERT preentrenado** y solo cambiaron la cabeza y los datos: cabeza siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta por token con CoNLL-2002 (C). Cada modelo afinado se aplicó **sobre su propio corpus**, y entre entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.


## Entregables — Lab 6
- [ ] **A:** fine-tuning de Sentence-BERT + los tres nDCG + búsqueda con el modelo afinado.
- [ ] **B:** clasificación con dataset real + matriz de confusión + inferencia sobre el corpus.
- [ ] **C:** NER con CoNLL-2002 + alineación de subpalabras + seqeval + extracción sobre el corpus.
- [ ] Celdas de liberación de memoria ejecutadas entre partes.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
